# Notebook 1: Data Collection from PDB

This notebook downloads protein structures from the RCSB Protein Data Bank,
runs DSSP to assign secondary structure labels, and saves a clean dataset.

### Design decisions
- **Source:** X-ray crystallography structures only (highest resolution)
- **Diversity:** Sampled from PDB's own 30% sequence identity clusters — one representative per cluster ensures proteins in the dataset are genuinely different from each other
- **One chain per entry:** Only the longest valid chain is kept per PDB entry, preventing multi-chain complexes from dominating the dataset
- **3-state labels:** DSSP's 8-state scheme (H, G, I, E, B, T, S, C) is mapped to 3 states: Helix (H), Strand (E), Coil (C)


In [ ]:
import os
import random
import requests
import pandas as pd
from Bio.PDB import PDBParser, DSSP

# ── Settings ──────────────────────────────────────────────────────────────────
DOWNLOAD_FOLDER = "pdb_files"
OUTPUT_CSV      = "protein_sequences_ss.csv"
NUM_PROTEINS    = 5000   # clusters to sample; actual yield will be lower after filtering
MIN_LEN         = 50     # minimum chain length (residues)
MAX_LEN         = 1000   # maximum chain length (residues)
RANDOM_SEED     = 42

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)


In [ ]:
# ── Fetch PDB's pre-clustered representative list ─────────────────────────────
# The RCSB maintains sequence identity clusters at 30%, 40%, 50%, 70%, 90%, 95%, 100%.
# Using 30% gives the most diverse set: ~258k clusters, each represented by one entry.
r = requests.get("https://cdn.rcsb.org/resources/sequence/clusters/clusters-by-entity-30.txt")
lines = r.text.strip().split("\n")
print(f"Total 30% identity clusters in PDB: {len(lines)}")

random.seed(RANDOM_SEED)
sampled_lines = random.sample(lines, min(NUM_PROTEINS, len(lines)))

# Each line looks like: "101M_1 102M_1 ..."
# We take the first token (representative), strip entity ID to get PDB ID
pdb_ids = []
for line in sampled_lines:
    rep    = line.strip().split()[0]
    pdb_id = rep.split("_")[0]
    if pdb_id.startswith("AF"):   # skip AlphaFold computational models
        continue
    pdb_ids.append(pdb_id)

# Multiple entities in the same entry map to the same PDB ID — deduplicate
pdb_ids = list(dict.fromkeys(pdb_ids))
print(f"Unique PDB entries to fetch: {len(pdb_ids)}")


In [ ]:
# ── Download, parse, and label ────────────────────────────────────────────────
parser         = PDBParser(QUIET=True)
data           = []
seen_sequences = set()   # exact-sequence deduplication across entries

VALID_AA  = set("ACDEFGHIKLMNPQRSTVWY")
SS_3STATE = lambda s: 'H' if s in "HGI" else ('E' if s in "EB" else 'C')

for pdb_id in pdb_ids:
    pdb_file = os.path.join(DOWNLOAD_FOLDER, f"{pdb_id}.pdb")

    # Download if not cached locally
    if not os.path.exists(pdb_file):
        url  = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        resp = requests.get(url)
        if resp.status_code == 200:
            with open(pdb_file, "w") as f:
                f.write(resp.text)
        else:
            print(f"  ✗ Download failed: {pdb_id}")
            continue

    try:
        structure = parser.get_structure(pdb_id, pdb_file)
        model     = structure[0]
        dssp      = DSSP(model, pdb_file, dssp="mkdssp")

        # Pick the single longest valid chain per PDB entry.
        # This prevents large complexes (e.g. ribosomes) from contributing
        # dozens of chains and dominating the dataset.
        best_chain = None

        for chain in model:
            chain_id = chain.id
            keys     = sorted([k for k in dssp.keys() if k[0] == chain_id],
                               key=lambda x: x[1])
            if not keys:
                continue

            seq     = "".join(dssp[k][1] for k in keys)
            ss_full = "".join(dssp[k][2] for k in keys)

            # Filter non-standard residues
            seq_clean = "".join(aa for aa in seq if aa in VALID_AA)
            ss_clean  = "".join(s  for aa, s in zip(seq, ss_full) if aa in VALID_AA)

            if not (MIN_LEN <= len(seq_clean) <= MAX_LEN):
                continue

            ss_simple = "".join(SS_3STATE(s) for s in ss_clean)
            if len(seq_clean) != len(ss_simple):
                continue

            if best_chain is None or len(seq_clean) > len(best_chain[0]):
                best_chain = (seq_clean, ss_simple, chain_id)

        if best_chain is None:
            continue

        sequence, ss_simple, chain_id = best_chain
        full_id = f"{pdb_id}_{chain_id}"

        if sequence in seen_sequences:
            continue
        seen_sequences.add(sequence)

        data.append({"pdb_id": full_id, "sequence": sequence,
                     "secondary_structure": ss_simple, "length": len(sequence)})
        print(f"  ✓ {full_id}  ({len(sequence)} residues)")

    except Exception as e:
        print(f"  ✗ {pdb_id}: {e}")

df = pd.DataFrame(data)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(df)} proteins → {OUTPUT_CSV}")


In [ ]:
# ── Dataset summary ───────────────────────────────────────────────────────────
import collections
import matplotlib.pyplot as plt

print(df.describe())
print(f"\nLength range: {df['length'].min()}–{df['length'].max()} residues")

# Secondary structure composition
all_ss = "".join(df["secondary_structure"])
counts = collections.Counter(all_ss)
total  = len(all_ss)
for state, n in sorted(counts.items()):
    print(f"  {state}: {n:,}  ({100*n/total:.1f}%)")

# Length distribution
plt.figure(figsize=(8, 3))
plt.hist(df["length"], bins=40, color="steelblue", edgecolor="white")
plt.xlabel("Chain length (residues)")
plt.ylabel("Count")
plt.title("Distribution of protein chain lengths")
plt.tight_layout()
plt.savefig("length_distribution.png", dpi=150)
plt.show()
